# Preflight & Viability Checks

Every shot you submit to quantum hardware costs money. If your circuit's estimated fidelity is below the noise floor, the output is indistinguishable from random noise: and the entire job is wasted QPU time.

`check_viability()` answers the question **before** you spend a cent: *will this circuit produce meaningful results on this backend?*

This notebook covers:

1. Running `check_viability()` on different circuit types
2. Understanding all three verdicts: **VIABLE**, **MARGINAL**, **NOT VIABLE**
3. Reading `ViabilityResult` fields in depth
4. Batch-checking multiple circuits
5. Adjusting the `viability_threshold` parameter
6. Building a preflight wrapper function
7. Comparing viability across backends
8. When to trust MARGINAL vs when to skip

## 1. Setup and Imports

In [1]:
import math

from qiskit import QuantumCircuit

from qb_compiler import check_viability, ViabilityResult

## 2. Your First Viability Check

Let's start with the simplest possible circuit: a 2-qubit Bell state. This should be comfortably VIABLE on any backend.

In [2]:
# Build a Bell state circuit
bell = QuantumCircuit(2, 2, name="Bell")
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])

result = check_viability(bell, backend="ibm_fez")
print(result)

/home/dministrator/miniconda3/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Circuit: Bell
Backend: ibm_fez
Estimated fidelity: 0.9430 (+-0.05 typical, model tends to overestimate; validated on n=6 hw circuits (Fez, GHZ family))
Noise floor: 0.2500
Signal/noise ratio: 3.8x
Status: VIABLE
Reason: Estimated fidelity 0.943 is well above noise floor (0.2500). Circuit should produce meaningful results.
Calibration snapshot age: 90 days
Error budget:
  - readout: 0.0559 (98% of loss)
  - two_qubit_gates: 0.0012 (2% of loss)
Cost (4096 shots): $0.6554
Suggestions:
  - Circuit looks good: proceed with execution.
  - Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.


The `print()` gives a human-readable summary. Let's look at the individual fields.

In [3]:
print(f"Status:             {result.status}")
print(f"Viable (bool):      {result.viable}")
print(f"Estimated fidelity: {result.estimated_fidelity:.4f}")
print(f"Noise floor:        {result.noise_floor:.4f}")
print(f"Signal-to-noise:    {result.signal_to_noise:.1f}x")
print(f"Depth:              {result.depth}")
print(f"2Q gates:           {result.two_qubit_gate_count}")
print(f"Viable depth:       {result.viable_depth}")
print(f"Cost (4096 shots):  ${result.cost_estimate_usd:.4f}")
print(f"Reason:             {result.reason}")
print(f"Suggestions:        {result.suggestions}")

Status:             VIABLE
Viable (bool):      True
Estimated fidelity: 0.9430
Noise floor:        0.2500
Signal-to-noise:    3.8x
Depth:              4
2Q gates:           1
Viable depth:       134
Cost (4096 shots):  $0.6554
Reason:             Estimated fidelity 0.943 is well above noise floor (0.2500). Circuit should produce meaningful results.
Suggestions:        ['Circuit looks good: proceed with execution.', 'Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.']


## 3. The Three Verdicts

| Status | Meaning | Action |
|--------|---------|--------|
| **VIABLE** | Fidelity >= 30% and well above noise floor | Proceed with execution |
| **MARGINAL** | Above noise floor but fidelity < 30% | Consider error mitigation (ZNE, PEC) |
| **NOT VIABLE** | Output is indistinguishable from noise | Do not submit: you will waste money |

Let's build circuits that trigger each verdict.

In [4]:
# VIABLE: Bell state (shallow, few gates)
bell = QuantumCircuit(2, 2, name="Bell")
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])

r_viable = check_viability(bell, backend="ibm_fez")
print(f"Bell state:     {r_viable.status}  (fidelity={r_viable.estimated_fidelity:.4f})")

Bell state:     VIABLE  (fidelity=0.9430)


In [5]:
# MARGINAL: medium-depth circuit with enough gates to erode fidelity
medium = QuantumCircuit(6, 6, name="MediumDepth")
for layer in range(30):
    for i in range(5):
        medium.cx(i, i + 1)
    for i in range(6):
        medium.rz(0.3, i)
medium.measure(range(6), range(6))

r_marginal = check_viability(medium, backend="ibm_fez")
print(f"Medium circuit: {r_marginal.status}  (fidelity={r_marginal.estimated_fidelity:.4f})")

Medium circuit: VIABLE  (fidelity=0.6300)


In [6]:
# NOT VIABLE: deep circuit that's buried in noise
deep = QuantumCircuit(4, 4, name="TooDeep")
for _ in range(200):
    for i in range(3):
        deep.cx(i, i + 1)
        deep.rz(0.5, i + 1)
deep.measure(range(4), range(4))

r_not_viable = check_viability(deep, backend="ibm_fez")
print(f"Deep circuit:   {r_not_viable.status}  (fidelity={r_not_viable.estimated_fidelity:.6f})")
print()
for s in r_not_viable.suggestions:
    print(f"  -> {s}")

Deep circuit:   VIABLE  (fidelity=0.321607)

  -> Reduce circuit depth from 1005 to <406 (current depth exceeds viable limit for this backend).
  -> Good candidate for error mitigation to further improve results.
  -> Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.


## 4. Checking Different Circuit Types

Different algorithmic circuit families have very different viability profiles. Let's build a GHZ state, a QFT, a QAOA-style circuit, and a VQE ansatz.

In [7]:
# GHZ state: linear chain of CX gates
def make_ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"GHZ-{n}")
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure(range(n), range(n))
    return qc


# QFT circuit: all-to-all controlled rotations
def make_qft(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"QFT-{n}")
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            qc.cp(math.pi / (2 ** (j - i)), i, j)
    qc.measure(range(n), range(n))
    return qc


# QAOA-style circuit: alternating mixer and problem layers
def make_qaoa(n: int, p: int = 2) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"QAOA-{n}-p{p}")
    for q in range(n):
        qc.h(q)
    for _ in range(p):
        # Problem layer (ZZ interactions on a ring)
        for i in range(n):
            j = (i + 1) % n
            qc.cx(i, j)
            qc.rz(0.5, j)
            qc.cx(i, j)
        # Mixer layer
        for i in range(n):
            qc.rx(0.7, i)
    qc.measure(range(n), range(n))
    return qc


# VQE-style ansatz: hardware-efficient alternating layers
def make_vqe_ansatz(n: int, layers: int = 3) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f"VQE-{n}-L{layers}")
    for layer in range(layers):
        for q in range(n):
            qc.ry(0.5 + layer * 0.1, q)
            qc.rz(0.3 + layer * 0.2, q)
        for q in range(0, n - 1, 2):
            qc.cx(q, q + 1)
        for q in range(1, n - 1, 2):
            qc.cx(q, q + 1)
    qc.measure(range(n), range(n))
    return qc


print("Circuit families built.")

Circuit families built.


In [8]:
# Check viability for each at a moderate qubit count
n = 8
circuits = [
    make_ghz(n),
    make_qft(n),
    make_qaoa(n, p=2),
    make_vqe_ansatz(n, layers=3),
]

print(f"{'Circuit':<20s} {'Status':<12s} {'Fidelity':>10s} {'2Q Gates':>10s} {'Depth':>8s} {'S/N':>8s}")
print("-" * 72)
for circ in circuits:
    r = check_viability(circ, backend="ibm_fez")
    print(
        f"{r.circuit_name:<20s} {r.status:<12s} "
        f"{r.estimated_fidelity:>10.4f} {r.two_qubit_gate_count:>10d} "
        f"{r.depth:>8d} {r.signal_to_noise:>8.1f}"
    )

Circuit              Status         Fidelity   2Q Gates    Depth      S/N
------------------------------------------------------------------------


GHZ-8                VIABLE           0.8604          7       16    220.3


QFT-8                VIABLE           0.7066         91      194    180.9


QAOA-8-p2            VIABLE           0.7269         56      152    186.1


VQE-8-L3             VIABLE           0.8364         21       28    214.1


Notice the pattern: GHZ and VQE ansatz circuits stay viable at moderate sizes because they have relatively few 2Q gates. QFT and QAOA circuits accumulate more 2Q gates and hit the noise floor faster.

## 5. Reading ViabilityResult Fields

Let's dissect every field of the result object.

In [9]:
# Pick a MARGINAL result for the most interesting field values
circ = make_qaoa(10, p=3)
r = check_viability(circ, backend="ibm_fez")

print("=== ViabilityResult Fields ===")
print()
print(f"viable:               {r.viable}")
print(f"  Boolean: True for VIABLE and MARGINAL, False for NOT VIABLE.")
print()
print(f"status:               {r.status}")
print(f"  Human-readable verdict string.")
print()
print(f"estimated_fidelity:   {r.estimated_fidelity:.6f}")
print(f"  Multiplicative fidelity from calibration data. Accounts for")
print(f"  per-gate errors and readout errors after transpilation.")
print()
print(f"noise_floor:          {r.noise_floor:.6f}")
print(f"  Probability of correct answer from random guessing (1/2^n).")
print()
print(f"signal_to_noise:      {r.signal_to_noise:.1f}x")
print(f"  estimated_fidelity / noise_floor. Below 2.0 means signal is")
print(f"  buried in noise.")
print()
print(f"two_qubit_gate_count: {r.two_qubit_gate_count}")
print(f"  Number of 2Q gates after transpilation (routing adds SWAPs).")
print()
print(f"depth:                {r.depth}")
print(f"  Circuit depth after transpilation.")
print()
print(f"viable_depth:         {r.viable_depth}")
print(f"  Maximum circuit depth that still produces useful results on")
print(f"  this backend at current error rates.")
print()
print(f"cost_estimate_usd:    ${r.cost_estimate_usd:.4f}")
print(f"  Estimated cost for a 4096-shot run.")
print()
print(f"reason:               {r.reason}")
print()
print(f"suggestions:")
for s in r.suggestions:
    print(f"  - {s}")

=== ViabilityResult Fields ===

viable:               True
  Boolean: True for VIABLE and MARGINAL, False for NOT VIABLE.

status:               VIABLE
  Human-readable verdict string.

estimated_fidelity:   0.675098
  Multiplicative fidelity from calibration data. Accounts for
  per-gate errors and readout errors after transpilation.

noise_floor:          0.000977
  Probability of correct answer from random guessing (1/2^n).

signal_to_noise:      691.3x
  estimated_fidelity / noise_floor. Below 2.0 means signal is
  buried in noise.

two_qubit_gate_count: 78
  Number of 2Q gates after transpilation (routing adds SWAPs).

depth:                226
  Circuit depth after transpilation.

viable_depth:         898
  Maximum circuit depth that still produces useful results on
  this backend at current error rates.

cost_estimate_usd:    $0.6554
  Estimated cost for a 4096-shot run.

reason:               Estimated fidelity 0.675 is well above noise floor (0.0010). Circuit should produce m

## 6. Batch Checking Multiple Circuits

When planning an experiment with multiple circuits (e.g. a VQE sweep), check them all at once to identify which will produce useful data.

In [10]:
# Sweep GHZ circuits from 2 to 30 qubits
results = []
for n in range(2, 31, 2):
    ghz = make_ghz(n)
    r = check_viability(ghz, backend="ibm_fez")
    results.append(r)

print(f"{'Qubits':>8s} {'Status':<12s} {'Fidelity':>10s} {'S/N':>8s} {'Depth':>8s} {'Viable Depth':>13s}")
print("-" * 65)
for r in results:
    qubits = int(r.circuit_name.split("-")[1])
    print(
        f"{qubits:>8d} {r.status:<12s} {r.estimated_fidelity:>10.4f} "
        f"{r.signal_to_noise:>8.1f} {r.depth:>8d} {r.viable_depth:>13d}"
    )

  Qubits Status         Fidelity      S/N    Depth  Viable Depth
-----------------------------------------------------------------
       2 VIABLE           0.9430      3.8        4           134
       4 VIABLE           0.8771     14.0        8           406
       6 VIABLE           0.8405     53.8       12           679
       8 VIABLE           0.8604    220.3       16           902
      10 VIABLE           0.7702    788.6       20           898
      12 VIABLE           0.7478   3063.2       24           894
      14 VIABLE           0.7116  11658.4       28           890
      16 VIABLE           0.7043  46157.4       32           886
      18 VIABLE           0.6597 172942.0       36           882
      20 VIABLE           0.6097 639350.2       40           878
      22 VIABLE           0.5771 2420360.7       44           874
      24 VIABLE           0.5656 9489010.0       48           870
      26 VIABLE           0.5077 34070873.8       52           866
      28 VIABLE     

In [11]:
# Summarize: at what qubit count does GHZ become non-viable?
viable_results = [r for r in results if r.status == "VIABLE"]
marginal_results = [r for r in results if r.status == "MARGINAL"]
not_viable_results = [r for r in results if r.status == "NOT VIABLE"]

print(f"VIABLE:     {len(viable_results)} circuits")
print(f"MARGINAL:   {len(marginal_results)} circuits")
print(f"NOT VIABLE: {len(not_viable_results)} circuits")

if marginal_results:
    first_marginal = marginal_results[0].circuit_name
    print(f"\nFirst MARGINAL: {first_marginal}")
if not_viable_results:
    first_nv = not_viable_results[0].circuit_name
    print(f"First NOT VIABLE: {first_nv}")

VIABLE:     15 circuits
MARGINAL:   0 circuits
NOT VIABLE: 0 circuits


## 7. Adjusting the Viability Threshold

The `viability_threshold` parameter controls the minimum signal-to-noise ratio required for a circuit to be considered viable. The default is 2.0 (fidelity must be at least 2x the noise floor).

Raising the threshold makes the check more conservative. Lowering it makes it more permissive.

In [12]:
circ = make_qaoa(8, p=2)

print(f"{'Threshold':>10s} {'Status':<12s} {'Fidelity':>10s} {'S/N':>8s}")
print("-" * 45)
for threshold in [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
    r = check_viability(circ, backend="ibm_fez", viability_threshold=threshold)
    print(f"{threshold:>10.1f} {r.status:<12s} {r.estimated_fidelity:>10.4f} {r.signal_to_noise:>8.1f}")

 Threshold Status         Fidelity      S/N
---------------------------------------------


       1.0 VIABLE           0.7269    186.1


       1.5 VIABLE           0.7269    186.1


       2.0 VIABLE           0.7269    186.1


       3.0 VIABLE           0.7269    186.1


       5.0 VIABLE           0.7269    186.1


      10.0 VIABLE           0.7269    186.1


**Guidance:**
- Use `threshold=2.0` (default) for production workloads where you need reliable results.
- Use `threshold=1.5` if you plan to apply error mitigation (ZNE, PEC).
- Use `threshold=5.0+` for research benchmarking where you need high confidence.

## 8. Comparing Viability Across Backends

The same circuit can be viable on one backend and not viable on another, because backends have different gate error rates, qubit counts, and coupling topologies.

In [13]:
circ = make_qaoa(6, p=3)

backends = ["ibm_fez", "ibm_torino", "rigetti_ankaa", "ionq_aria", "iqm_garnet"]

print(f"Circuit: {circ.name}")
print()
print(f"{'Backend':<18s} {'Status':<12s} {'Fidelity':>10s} {'S/N':>8s} {'2Q Gates':>10s} {'Cost':>10s}")
print("-" * 72)
for backend in backends:
    r = check_viability(circ, backend=backend)
    cost_str = f"${r.cost_estimate_usd:.4f}" if r.cost_estimate_usd else "N/A"
    print(
        f"{backend:<18s} {r.status:<12s} {r.estimated_fidelity:>10.4f} "
        f"{r.signal_to_noise:>8.1f} {r.two_qubit_gate_count:>10d} {cost_str:>10s}"
    )

Circuit: QAOA-6-p3

Backend            Status         Fidelity      S/N   2Q Gates       Cost
------------------------------------------------------------------------


ibm_fez            VIABLE           0.7673     49.1         65    $0.6554


ibm_torino         VIABLE           0.5570     35.6         80    $0.5734


rigetti_ankaa      MARGINAL         0.1800     11.5         80    $1.4336
ionq_aria          VIABLE           0.7910     50.6         54  $122.8800
iqm_garnet         VIABLE           0.3917     25.1         54    $1.8432


## 9. Building a Preflight Wrapper

In production, wrap every hardware submission with a viability check. This function checks viability and only returns the circuit if it's worth running.

In [14]:
def preflight_check(
    circuit: QuantumCircuit,
    backend: str,
    *,
    allow_marginal: bool = True,
    viability_threshold: float = 2.0,
    max_cost_usd: float | None = None,
) -> tuple[bool, ViabilityResult]:
    """Preflight check: returns (should_run, result).

    Parameters
    ----------
    circuit : QuantumCircuit
        Circuit to check.
    backend : str
        Target backend name.
    allow_marginal : bool
        If False, MARGINAL circuits are rejected.
    viability_threshold : float
        Minimum signal-to-noise ratio.
    max_cost_usd : float, optional
        Maximum acceptable cost for a 4096-shot run.

    Returns
    -------
    tuple[bool, ViabilityResult]
        (should_run, viability_result)
    """
    result = check_viability(
        circuit,
        backend=backend,
        viability_threshold=viability_threshold,
    )

    # Check viability status
    if result.status == "NOT VIABLE":
        return False, result
    if result.status == "MARGINAL" and not allow_marginal:
        return False, result

    # Check cost budget
    if max_cost_usd is not None and result.cost_estimate_usd is not None:
        if result.cost_estimate_usd > max_cost_usd:
            return False, result

    return True, result


print("preflight_check() defined.")

preflight_check() defined.


In [15]:
# Use the preflight wrapper
circuits_to_run = [
    make_ghz(4),
    make_qaoa(12, p=4),
    make_vqe_ansatz(6, layers=2),
    make_qft(10),
]

approved = []
rejected = []

for circ in circuits_to_run:
    should_run, result = preflight_check(circ, "ibm_fez", allow_marginal=False)
    if should_run:
        approved.append(result)
        print(f"  APPROVED: {result.circuit_name} (fidelity={result.estimated_fidelity:.4f})")
    else:
        rejected.append(result)
        print(f"  REJECTED: {result.circuit_name} ({result.status}, fidelity={result.estimated_fidelity:.4f})")

print(f"\nApproved: {len(approved)} / {len(circuits_to_run)}")
print(f"Rejected: {len(rejected)} / {len(circuits_to_run)}")

if rejected:
    total_saved = sum(r.cost_estimate_usd for r in rejected if r.cost_estimate_usd)
    print(f"Money saved by skipping rejected circuits: ${total_saved:.4f}")

  APPROVED: GHZ-4 (fidelity=0.8771)


  APPROVED: QAOA-12-p4 (fidelity=0.5866)


  APPROVED: VQE-6-L2 (fidelity=0.8322)


  APPROVED: QFT-10 (fidelity=0.6113)

Approved: 4 / 4
Rejected: 0 / 4


## 10. When to Trust MARGINAL

A MARGINAL verdict means the circuit *can* produce meaningful results, but the signal is weak. Whether to proceed depends on your situation:

**Proceed with MARGINAL if:**
- You plan to apply error mitigation (ZNE, PEC, M3/mthree)
- You need relative comparisons rather than absolute fidelity
- The circuit is part of a research study exploring noise limits

**Skip MARGINAL if:**
- You need high-confidence results for a production application
- You're on a tight budget and can't afford wasted shots
- The suggestions recommend a simulator instead

Let's demonstrate the boundary between MARGINAL and NOT VIABLE.

In [16]:
# Find the QAOA depth where IBM Fez transitions from MARGINAL to NOT VIABLE
print(f"{'QAOA p':>8s} {'Status':<12s} {'Fidelity':>10s} {'S/N':>8s} {'Suggestion':>50s}")
print("-" * 95)

for p in range(1, 12):
    circ = make_qaoa(8, p=p)
    r = check_viability(circ, backend="ibm_fez")
    suggestion = r.suggestions[0][:47] + "..." if len(r.suggestions[0]) > 50 else r.suggestions[0]
    print(
        f"{p:>8d} {r.status:<12s} {r.estimated_fidelity:>10.4f} "
        f"{r.signal_to_noise:>8.1f} {suggestion:>50s}"
    )

  QAOA p Status         Fidelity      S/N                                         Suggestion
-----------------------------------------------------------------------------------------------


       1 VIABLE           0.7353    188.2        Circuit looks good: proceed with execution.


       2 VIABLE           0.7269    186.1        Circuit looks good: proceed with execution.


       3 VIABLE           0.6541    167.4        Circuit looks good: proceed with execution.


       4 VIABLE           0.6055    155.0        Circuit looks good: proceed with execution.


...

       7 VIABLE           0.5121    131.1        Circuit looks good: proceed with execution.


       8 VIABLE           0.4502    115.2 Good candidate for error mitigation to further ...


       9 VIABLE           0.4288    109.8 Good candidate for error mitigation to further ...


      10 VIABLE           0.4058    103.9 Good candidate for error mitigation to further ...


      11 VIABLE           0.3811     97.6 Good candidate for error mitigation to further ...


## 11. Random Circuits and Worst-Case Analysis

Random circuits are useful for stress-testing viability because they tend to have high 2Q gate density.

In [17]:
import random

random.seed(42)


def make_random_circuit(n: int, depth: int) -> QuantumCircuit:
    """Build a random circuit with the given number of qubits and depth."""
    qc = QuantumCircuit(n, n, name=f"Random-{n}q-d{depth}")
    for _ in range(depth):
        # Random single-qubit gate on each qubit
        for q in range(n):
            gate = random.choice(["h", "x", "rz"])
            if gate == "rz":
                qc.rz(random.uniform(0, 2 * math.pi), q)
            elif gate == "h":
                qc.h(q)
            else:
                qc.x(q)
        # Random CX on a random pair
        q0, q1 = random.sample(range(n), 2)
        qc.cx(q0, q1)
    qc.measure(range(n), range(n))
    return qc


# Sweep random circuit depth
print(f"{'Depth':>8s} {'Status':<12s} {'Fidelity':>10s} {'2Q Gates':>10s}")
print("-" * 44)

for d in [5, 10, 20, 50, 100, 200]:
    rc = make_random_circuit(6, d)
    r = check_viability(rc, backend="ibm_fez")
    print(f"{d:>8d} {r.status:<12s} {r.estimated_fidelity:>10.4f} {r.two_qubit_gate_count:>10d}")

   Depth Status         Fidelity   2Q Gates
--------------------------------------------


       5 VIABLE           0.8480          6


      10 VIABLE           0.8396         16


      20 VIABLE           0.8016         41


      50 VIABLE           0.6973        119


     100 VIABLE           0.5425        245


     200 VIABLE           0.3466        500


## 12. Using the `n_seeds` Parameter

Viability uses Qiskit's transpiler to estimate the routed circuit's gate count. More seeds means a better routing result but slower analysis.

In [18]:
import time

circ = make_qaoa(8, p=2)

print(f"{'Seeds':>8s} {'2Q Gates':>10s} {'Fidelity':>10s} {'Time (s)':>10s}")
print("-" * 42)

for seeds in [1, 5, 10, 20]:
    t0 = time.perf_counter()
    r = check_viability(circ, backend="ibm_fez", n_seeds=seeds)
    elapsed = time.perf_counter() - t0
    print(f"{seeds:>8d} {r.two_qubit_gate_count:>10d} {r.estimated_fidelity:>10.4f} {elapsed:>10.2f}")

   Seeds   2Q Gates   Fidelity   Time (s)
------------------------------------------


       1         56     0.7269       0.20


       5         56     0.7269       0.94


      10         56     0.7269       2.21


      20         56     0.7269       5.56


More seeds generally find a better routing (fewer 2Q gates), which can push a MARGINAL circuit into VIABLE territory. For quick screening, `n_seeds=5` is a good trade-off. For final go/no-go decisions, use `n_seeds=20`.

## Summary

In this notebook you learned:

- **`check_viability()`**: the single function that tells you whether a circuit is worth running
- **Three verdicts**: VIABLE (go), MARGINAL (go with caution), NOT VIABLE (stop)
- **`ViabilityResult` fields**: `estimated_fidelity`, `signal_to_noise`, `depth`, `viable_depth`, `cost_estimate_usd`, `suggestions`
- **Batch checking**: sweep qubit counts or circuit depths to find viability limits
- **`viability_threshold`**: tune the conservatism of the check (default 2.0)
- **Preflight wrapper**: production pattern for gating hardware submissions
- **Backend comparison**: the same circuit can have different viability across backends
- **MARGINAL guidance**: when to proceed and when to skip

Next: [02: Compilation & Audit Receipts](02_compilation_receipts.ipynb)